Homework: https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/05-monitoring/homework.md

# Download resoures

In [5]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring"
!wget -nc {PREFIX}/rag_helper.py
!wget -nc {PREFIX}/starter.py

File 'rag_helper.py' already there; not retrieving.

File 'starter.py' already there; not retrieving.



# Setup OpenAI Client

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from pathlib import Path

path = Path.cwd().parent/".env"
print(load_dotenv(path))

llm_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
    )

True


# Test RAG

Update `client=OpenAI()` -> 

```python
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
    )
```
 and add `model="openai/gpt-oss-120b` in `RAGBase()`

In [2]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The agentic loop is just a **while‑True loop that repeats the call to the LLM until the model returns a response that contains no tool (function) calls**.  
Here’s how it works step‑by‑step (taken from the lesson *14‑agentic‑loop.md*):

1. **Initialize the conversation history** (`messages`) with the developer prompt (instructions) and the user’s question.

2. **Enter a `while True` loop** and keep an iteration counter (`it`) so you can see how many round‑trips have happened.

3. **Call the model** with the current `messages` and the list of available tools (e.g., `search_tool`):

   ```python
   response = openai_client.responses.create(
       model="gpt-5.4-mini",
       input=messages,
       tools=[search_tool],
   )
   ```

4. **Append the model’s output** (both messages and any function‑call objects) to `messages` so the next turn sees everything that has happened so far.

5. **Inspect the output items**:
   - If an item’s `type` is `"function_call"`, the model is asking you to 

# OpenTelemetry Setup

Example 1

In [4]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")

Overriding of current TracerProvider is not allowed


{
    "name": "my_operation",
    "context": {
        "trace_id": "0x1bf8611f50767792f123302ad845f0e0",
        "span_id": "0x2a761331f90a34a9",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T04:26:57.758916Z",
    "end_time": "2026-07-20T04:26:57.759087Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "d10cdaf0-331d-45ea-8b45-7e9c5c29be41",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


Example 2

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# 1. Initialize the OpenTelemetry infrastructure
# Create the central configuration provider
provider = TracerProvider()

# Wire up a simple span processor that prints every finished span to the console
processor = SimpleSpanProcessor(ConsoleSpanExporter())
provider.add_span_processor(processor)

# Register the provider globally so other modules can access it
trace.set_tracer_provider(provider)

# 2. Get a tracer instance for your app
tracer = trace.get_tracer("my-traced-application")

# 3. Execute the traced code block
print("--- Starting operation ---")
with tracer.start_as_current_span("my_operation") as span:
    # Your business logic lives inside this block
    print("Performing some work inside the span...")
    
    # Attach custom metrics or metadata to this specific span
    span.set_attribute("my_key", "my_value")
    span.set_attribute("user_id", 42)

print("--- Operation finished! Check console output below ---")

Overriding of current TracerProvider is not allowed


--- Starting operation ---
Performing some work inside the span...
{
    "name": "my_operation",
    "context": {
        "trace_id": "0x6b9ed543496a3f1d94d38ff1a6c3a7d8",
        "span_id": "0x180bffe1dbd83e24",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T04:24:14.091170Z",
    "end_time": "2026-07-20T04:24:14.091281Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value",
        "user_id": 42
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "d10cdaf0-331d-45ea-8b45-7e9c5c29be41",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
--- Operation finished! Check console output below ---


# Q1. First trace

In [5]:
import time

from tqdm.auto import tqdm
from rag_helper import RAGBase

In [19]:
class RAGTraced(RAGBase):

    def rag(self, query):
        # Create the top-level "parent" span for the entire RAG pipeline
        with tracer.start_as_current_span("rag") as span:
            # Call the original rag method from the parent class
            return super().rag(query)

    def search(self, query, num_results=5):
        # Create a "child" span specifically for the index search step
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        # Create a "child" span specifically for the LLM API call step
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
             
            return response

In [27]:
# Q1) How many spans does the trace produce?

from starter import index

query = 'How does the agentic loop keep calling the model until it stops?'
assistant = RAGTraced(
    index=index,
    llm_client=llm_client,
    model="openai/gpt-oss-120b",
)
assistant.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0xd96f8d72037cbda8fe57188f04547e13",
        "span_id": "0x553bc7dc436271c2",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb73381ed7b41b61f",
    "start_time": "2026-07-20T05:49:08.308431Z",
    "end_time": "2026-07-20T05:49:08.326666Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "d10cdaf0-331d-45ea-8b45-7e9c5c29be41",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xd96f8d72037cbda8fe57188f04547e13",
        "span_id": "0x57920c346539eca9",
        "trace_state": "[]"
    },
    "kind": "SpanKind

'The agentic loop is just a **while‑True loop** that repeatedly sends the current message history to the model, looks at the response, and decides whether another round is needed.\n\n1. **Send the current `messages` list** to the model (`openai_client.responses.create`).  \n2. **Append the model’s output** (messages and any `function_call` entries) to the same `messages` list so the model sees everything it has already done.  \n3. **Inspect the response**:  \n   * If the response contains a `function_call`, the code runs the called tool (`make_call`), appends the tool’s result back into `messages`, and sets a flag `has_function_calls = True`.  \n   * If the response contains only normal `message` items, the flag stays `False`.  \n4. **Loop control** – after processing all items, the loop checks the flag:  \n\n```python\nif has_function_calls == False:\n    break        # no more tool calls → we are done\n```\n\nBecause the flag is only cleared when the model returns a turn **without an

Q1) How many spans does the trace produce?

- First Span: "name": `"search"` has trace_id: "0x899c38e5194973aecf24ee06bcaf7df3"

- Second Span: "name": `"llm"` has trace_id: "0x899c38e5194973aecf24ee06bcaf7df3"

- Third Span: "name": `"rag"` has trace_id: "0x899c38e5194973aecf24ee06bcaf7df3"

<font color='green'>Total spans by the trace: 3</font>

# Q2. Capturing metrics as span attributes


In [22]:
class RAGTraced(RAGBase):

    def rag(self, query):
        # Create the top-level "parent" span for the entire RAG pipeline
        with tracer.start_as_current_span("rag") as span:
            # Call the original rag method from the parent class
            return super().rag(query)

    def search(self, query, num_results=5):
        # Create a "child" span specifically for the index search step
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        # Create a "child" span specifically for the LLM API call step
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            # Q2. Capture token metrics from the response
            if hasattr(response, 'usage') and response.usage:
                span.set_attribute("input_tokens", response.usage.input_tokens)
                span.set_attribute("output_tokens", response.usage.output_tokens)
             
            return response

In [23]:
# Q2) How many input tokens do we see?

from starter import index

query = 'How does the agentic loop keep calling the model until it stops?'
assistant = RAGTraced(
    index=index,
    llm_client=llm_client,
    model="openai/gpt-oss-120b",
)
assistant.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0xdcfb3971043ce74d1ed6f1de9158676b",
        "span_id": "0xb6b19c3be1e801c7",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xdfddd10eb7b72bf9",
    "start_time": "2026-07-20T05:33:16.744486Z",
    "end_time": "2026-07-20T05:33:16.751598Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "d10cdaf0-331d-45ea-8b45-7e9c5c29be41",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xdcfb3971043ce74d1ed6f1de9158676b",
        "span_id": "0xa81510406ad0261b",
        "trace_state": "[]"
    },
    "kind": "SpanKind

'The agentic loop is just a tiny `while‑True` that repeatedly sends the current conversation to the model, runs any tool calls that the model asks for, and then checks whether the model still wants to call a tool.  \n\n**Step‑by‑step**\n\n1. **Start the loop** – `it = 1` and `while True:` begin an infinite loop.  \n2. **Call the model** – `openai_client.responses.create(..., input=messages, tools=[search_tool])` sends the whole message history (`messages`) plus the tool definitions.  \n3. **Append the model’s output** – `messages.extend(response.output)` adds everything the model returned (messages *and* any `function_call` entries) to the history.  \n4. **Process the output**  \n   * For each item in `response.output`  \n     * If `item.type == "function_call"` → the model is asking for a tool.  \n       * The helper `make_call(item)` runs the actual Python function (`search`), serialises the result, and appends a `function_call_output` entry back into `messages`.  \n       * `has_fun

 Q2. How many input tokens do we see?

```bash
 "attributes": {
        "input_tokens": 7174,
        "output_tokens": 624
    }
```
<font color='green'>Answer: ~7000 </font>
>

# Q3. Span timing

From Q1.

```bash
"name": "llm",
    "context": {
        "trace_id": "0x9762848f288bb536e0b36c5af5db392b",
        "span_id": "0x21bf5dbd57efd9ea",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2bbfdf59ad5b06e8",
    "start_time": "2026-07-20T05:29:54.550763Z",
    "end_time": "2026-07-20T05:29:56.696679Z",
```

In [26]:
from datetime import datetime

# Timestamps from your output
search_start = datetime.strptime("2026-07-20T05:29:54.542526Z", "%Y-%m-%dT%H:%M:%S.%fZ")
search_end = datetime.strptime("2026-07-20T05:29:54.548211Z", "%Y-%m-%dT%H:%M:%S.%fZ")

llm_start = datetime.strptime("2026-07-20T05:29:54.550763Z", "%Y-%m-%dT%H:%M:%S.%fZ")
llm_end = datetime.strptime("2026-07-20T05:29:56.696679Z", "%Y-%m-%dT%H:%M:%S.%fZ")

# Calculation
search_ms = (search_end - search_start).total_seconds() * 1000
llm_ms = (llm_end - llm_start).total_seconds() * 1000

print(f"Search span duration: {search_ms:.2f} ms")
print(f"LLM span duration: {llm_ms:.2f} ms")

Search span duration: 5.68 ms
LLM span duration: 2145.92 ms


Q3. For a typical query, roughly how long does the LLM call take?

<font color=green>Answer: > 2000ms </font>

# Q4. Saving traces to SQLite


Need to restart the notebook to resolve -> `Overriding of current TracerProvider is not allowed`

Declare traces.db :

In [3]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

traces.db store span:
- name
- start_time
- end_time
- input_tokens
- output_tokens
- cost

Set up tracer provider:

In [4]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()

# REPLACE ConsoleSpanExporter with SQLiteSpanExporter
processor = SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
provider.add_span_processor(processor)

# Register the provider globally so other modules can access it
trace.set_tracer_provider(provider)

# Get a tracer instance for your app
tracer = trace.get_tracer("llm-zoomcamp")


Initialize and run:

In [5]:
from rag_helper import RAGBase 

class RAGTraced(RAGBase):

    def rag(self, query):
        # Create the top-level "parent" span for the entire RAG pipeline
        with tracer.start_as_current_span("rag") as span:
            # Call the original rag method from the parent class
            return super().rag(query)

    def search(self, query, num_results=5):
        # Create a "child" span specifically for the index search step
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        # Create a "child" span specifically for the LLM API call step
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            # Q2. Capture token metrics from the response
            if hasattr(response, 'usage') and response.usage:
                span.set_attribute("input_tokens", response.usage.input_tokens)
                span.set_attribute("output_tokens", response.usage.output_tokens)

                # calculate cost
                input_price_per_million = 0.15 # groq opeanai/gpt-oss-120b # 0.75 
                output_price_per_million = 0.6 # groq opeanai/gpt-oss-120b # 4.50

                input_cost = (response.usage.input_tokens / 1_000_000) * input_price_per_million
                output_cost = (response.usage.output_tokens / 1_000_000) * output_price_per_million
                total_cost = input_cost + output_cost
                span.set_attribute("cost",total_cost)

            return response


# ---------------------- #
from starter import index

query = 'How does the agentic loop keep calling the model until it stops?'
assistant = RAGTraced(
    index=index,
    llm_client=llm_client,
    model="openai/gpt-oss-120b",
)
assistant.rag(query)

'The agentic loop is just a **while‑True loop that repeatedly sends the current conversation to the model, runs any tool calls the model requests, and then checks whether the model asked for another tool**.  \n\nHere’s the core idea from the lesson:\n\n1. **Start with the initial messages** (developer instructions + user question).  \n2. **Enter a `while True` loop**.  \n3. On each iteration:  \n   * Call the model (`openai_client.responses.create`) with the whole `messages` list and the tool definitions.  \n   * Append the model’s output (`response.output`) to `messages` so the model sees everything it has already said and any tool results.  \n   * Scan the output items:  \n     * If an item’s `type` is **`function_call`**, run the corresponding Python function (via `make_call`), append the function‑call output back into `messages`, and set a flag `has_function_calls = True`.  \n     * If the item’s `type` is **`message`**, display the assistant’s text (this may be the final answer). 

In [6]:
import sqlite3
import pandas as pd

# 1. Connect to .db
conn = sqlite3.connect("traces.db")

# 2. Query all rows to inspect what was captured
df = pd.read_sql_query("SELECT * FROM spans", conn)

# Close the connection
conn.close()

# Display the data frame
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784529949562570000,1784529949581253000,NaN,NaN,NaN
1,llm,1784529949582798000,1784529952335876000,7174.0,711.0,0.001503
2,rag,1784529949562286000,1784529952336733000,NaN,NaN,NaN


Q4. Which span names appear in the spans table?

<font color="green">Answer: search, llm, rag </font>

# Q5. Querying trace data

In [24]:
# ---------------------- #
from starter import index

query = 'How does the agentic loop keep calling the model until it stops?'
assistant = RAGTraced(
    index=index,
    llm_client=llm_client,
    model="openai/gpt-oss-120b",
)
assistant.rag(query)

'The agentic loop is just a **`while True` loop** that repeatedly:\n\n1. **Sends the current message history** (`messages`) to the model with the available tools.  \n2. **Appends the model’s output** (messages and any `function_call` objects) to the same `messages` list so the next turn sees everything that has happened.  \n3. **Scans the returned items**:  \n   * If an item’s `type` is `"function_call"` it runs the corresponding tool (via `make_call`), appends the tool’s result back into `messages`, and sets a flag `has_function_calls = True`.  \n   * If an item’s `type` is `"message"` it prints the assistant’s reply (and stores it as the final answer).  \n4. **Increments an iteration counter** (just for logging).  \n5. **Breaks out of the loop** when the flag `has_function_calls` is still `False` after processing the model’s response – meaning the model didn’t ask for any more tool calls, so it has produced a final answer.\n\n```python\nwhile True:\n    response = openai_client.respo

In [25]:
conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT * FROM spans", conn)
conn.close()
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784529949562570000,1784529949581253000,NaN,NaN,NaN
1,llm,1784529949582798000,1784529952335876000,7174.0,711.0,0.001503
2,rag,1784529949562286000,1784529952336733000,NaN,NaN,NaN
3,search,1784529977640466000,1784529977651873000,NaN,NaN,NaN
4,llm,1784529977652934000,1784530011039085000,7174.0,563.0,0.001414
5,rag,1784529977640361000,1784530011042601000,NaN,NaN,NaN
6,search,1784530613657178000,1784530613665969000,NaN,NaN,NaN
7,llm,1784530613669339000,1784530615650767000,7174.0,526.0,0.001392
8,rag,1784530613657095000,1784530615651775000,NaN,NaN,NaN


In [26]:
df = df[df['name'] != "rag"].sort_values('name')
df['duration_ms'] = (df['end_time'] - df['start_time']) / 1_000_000
df.drop(columns=['start_time','end_time','cost'],axis=0)

,name,input_tokens,output_tokens,duration_ms
1,llm,7174.0,711.0,2753.078
4,llm,7174.0,563.0,33386.151
7,llm,7174.0,526.0,1981.428
0,search,NaN,NaN,18.683
3,search,NaN,NaN,11.407
6,search,NaN,NaN,8.791


In [28]:
# Total duration by span type, excluding rag)
summary = df[df['name'] != 'rag'].groupby('name')['duration_ms'].sum().reset_index()
summary

,name,duration_ms
0,llm,38120.657
1,search,38.881


Q5) Which span type takes the most total time?

<font color="green">Answer: llm </font>

# Q6. Token stability across runs

In [36]:
# ---------------------- #
from starter import index

query = 'How does the agentic loop keep calling the model until it stops?'
assistant = RAGTraced(
    index=index,
    llm_client=llm_client,
    model="openai/gpt-oss-120b",
)
assistant.rag(query)

'The agentic loop is just a **`while‑True`** loop that repeatedly sends the current conversation history to the model, runs any tool calls the model requested, and then checks whether the model asked for another tool.  \n\nHere’s the flow:\n\n1. **Start a loop** (`while True:`) and set a flag `has_function_calls = False` at the beginning of each iteration.  \n2. **Call the model** with the full `messages` list (the developer prompt, the user question, plus every previous model message, tool call, and tool result).  \n3. **Append the model’s output** (`response.output`) to `messages` so the next turn sees everything that has happened so far.  \n4. **Iterate over the items** in `response.output`  \n   * If an item’s `type` is **`function_call`**, run the corresponding tool (via `make_call(item)`), append the tool’s result to `messages`, and set `has_function_calls = True`.  \n   * If the item is a normal **`message`**, just print/display it (this is the assistant’s answer for that turn).

In [37]:
conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT * FROM spans", conn)
conn.close()
df = df[df['name']=='llm']
df.drop(columns=['start_time','end_time'],axis=0)

,name,input_tokens,output_tokens,cost
1,llm,7174.0,711.0,0.001503
4,llm,7174.0,563.0,0.001414
7,llm,7174.0,526.0,0.001392
10,llm,9478.0,389.0,0.001655
13,llm,7174.0,542.0,0.001401
16,llm,7174.0,564.0,0.001414


Q6) How much do the input tokens vary across these 4 runs?

<font color="green">Answer: They're identical </font>